# Human Activity Recognition (HAR) — Interactive Validation Dashboard
### Sensor-Based Human Activity Recognition Using Deep Learning

**Provisional Best Model:** Targeted Class-Weighted 1D Convolutional Neural Network (CNN)  
**Input Signal Windows:** 128 timesteps (2.56 seconds at 50 Hz) across 6 inertial sensor channels (`body_acc_x/y/z`, `body_gyro_x/y/z`)  
**Target Activities (6):** `WALKING`, `WALKING_UPSTAIRS`, `WALKING_DOWNSTAIRS`, `SITTING`, `STANDING`, `LAYING`  
**Validation Scheme:** Participant-separated split (`GroupShuffleSplit`, `test_size=0.20`, `random_state=42`)  
**Operational Disclaimer:** Offline evaluation on pre-recorded benchmark windows (UCI HAR Dataset ID 240); **not live smartphone sensing**.

---

### Dashboard Key Objectives & Workflow:
1. **Model Selection & Safety Audit:** Verify prediction equivalence between the candidate checkpoints:
   - `02_Model_Training/outputs/targeted_sitting_experiment/best_targeted_cnn.keras`
   - `03_Final_Model/final_model/har_cnn_model_verified_candidate.keras`  
   Use the verified candidate checkpoint if equivalence is confirmed; otherwise use the targeted CNN directly. Confirm input shape `(None, 128, 6)` and 6 output classes without retraining, overwriting, renaming, or deleting any files.
2. **Dataset Partitioning:** Load preprocessed, normalized UCI HAR signals (`har_processed_data.npz`) and recreate the participant-separated validation split with zero participant leakage.
3. **Live Validation Metrics:** Directly evaluate the loaded model on all 1,801 validation windows to obtain live Accuracy, Macro Precision, Macro Recall, Macro F1, and Weighted F1.
4. **Interactive Dashboard UI:**
   - Validation sample dropdown selector (by sample index or activity category).
   - Interactive **Run Prediction** button.
   - Predicted activity, actual activity label, and top prediction confidence.
   - Distinct match/mismatch visual verdict badge.
   - Complete 6-class probability distribution bar chart.
   - Optional 6-channel raw inertial signal waveforms (Body Acc X/Y/Z and Body Gyro X/Y/Z).
   - Time-domain sample explanation (128 steps, 2.56 s at 50 Hz).


## 1. Environment Setup & Google Drive Mounting
Mounts Google Drive (if running in Google Colab), configures random seeds for exact reproducibility, and resolves project directory paths.


In [ ]:
# 1. Environment Setup & Google Drive Mounting
import os
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Suppress minor warnings for clean presentation
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Set random seed for full reproducibility
SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

# Mount Google Drive if in Google Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
    print("Google Colab runtime detected. Google Drive mounted successfully.")
except (ImportError, ModuleNotFoundError):
    IN_COLAB = False
    print("Running in standard / local Python runtime.")

# Robust Project Path Resolver
def resolve_project_root() -> Path:
    candidates = [
        Path("/content/drive/MyDrive/Human_Activity_Recognition"),
        Path("/content/drive/MyDrive/aiproject"),
        Path.cwd(),
        Path.cwd() / "Human_Activity_Recognition",
        Path.cwd().parent,
        Path.cwd().parent / "Human_Activity_Recognition",
    ]
    for candidate in candidates:
        if (candidate / "01_Dataset_Preprocessing/processed_data/har_processed_data.npz").exists():
            return candidate.resolve()
    # Default to standard Colab path
    return Path("/content/drive/MyDrive/Human_Activity_Recognition")

PROJECT_DIR = resolve_project_root()
DATA_PATH = PROJECT_DIR / "01_Dataset_Preprocessing/processed_data/har_processed_data.npz"
TARGETED_CHECKPOINT = PROJECT_DIR / "02_Model_Training/outputs/targeted_sitting_experiment/best_targeted_cnn.keras"
CANDIDATE_CHECKPOINT = PROJECT_DIR / "03_Final_Model/final_model/har_cnn_model_verified_candidate.keras"
OUTPUT_DIR = PROJECT_DIR / "03_Final_Model/dashboard_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fixed HAR dataset definitions
ACTIVITY_NAMES = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

CHANNEL_NAMES = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z"
]

SAMPLING_RATE_HZ = 50
WINDOW_SIZE = 128
WINDOW_DURATION_SEC = WINDOW_SIZE / SAMPLING_RATE_HZ  # 2.56 seconds

print("=" * 75)
print(f"TensorFlow Version:       {tf.__version__}")
print(f"Project Directory:        {PROJECT_DIR}")
print(f"Dataset Archive Exists:   {DATA_PATH.exists()} ({DATA_PATH})")
print(f"Targeted Checkpoint:      {TARGETED_CHECKPOINT.exists()} ({TARGETED_CHECKPOINT.name})")
print(f"Candidate Checkpoint:     {CANDIDATE_CHECKPOINT.exists()} ({CANDIDATE_CHECKPOINT.name})")
print(f"Dashboard Outputs Folder: {OUTPUT_DIR}")
print(f"Number of Classes:        {len(ACTIVITY_NAMES)}")
print(f"Sensor Channels (6):      {', '.join(CHANNEL_NAMES)}")
print(f"Window Duration:          {WINDOW_SIZE} steps = {WINDOW_DURATION_SEC:.2f} s @ {SAMPLING_RATE_HZ} Hz")
print("=" * 75)


## 2. Load Preprocessed Dataset & Recreate Validation Split
- Loads `har_processed_data.npz` directly from Google Drive / project files.
- Uses existing normalized 6-channel signals; **no secondary normalization is performed**.
- Recreates the participant-separated validation partition using `GroupShuffleSplit`, `test_size=0.20`, and `random_state=42`.
- Verifies that training and validation subjects are strictly disjoint (zero participant leakage).


In [ ]:
# 2. Load Preprocessed Dataset and Recreate Validation Split
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "CRITICAL ERROR: Preprocessed dataset archive not found at:\n"
        f"  {DATA_PATH}\n"
        "Please verify your Google Drive directory or project structure before proceeding."
    )

print("Loading preprocessed UCI HAR dataset archive...")
with np.load(DATA_PATH, allow_pickle=False) as npz_data:
    required_keys = {"X_train", "y_train", "subject_train", "activity_names", "channel_names"}
    missing_keys = required_keys - set(npz_data.files)
    if missing_keys:
        raise KeyError(f"Dataset archive is missing required arrays: {missing_keys}")

    X_train_full = npz_data["X_train"]
    y_train_full = npz_data["y_train"].astype(int)
    subject_train_full = npz_data["subject_train"].astype(int)

# Verify signal shapes and sanity
assert X_train_full.ndim == 3 and X_train_full.shape[1:] == (WINDOW_SIZE, len(CHANNEL_NAMES)), (
    f"Expected X_train shape (N, {WINDOW_SIZE}, {len(CHANNEL_NAMES)}), got {X_train_full.shape}"
)
assert np.isfinite(X_train_full).all(), "Signal array contains non-finite (NaN or Inf) values!"
assert set(np.unique(y_train_full)).issubset(set(range(len(ACTIVITY_NAMES)))), "Invalid class labels found."

print(f"Preprocessed training pool: {X_train_full.shape[0]} windows, shape {X_train_full.shape}")
print(f"All signal values finite:   {bool(np.isfinite(X_train_full).all())}")
print(f"Channels verified:          {CHANNEL_NAMES}")
print(f"Activity labels verified:   {ACTIVITY_NAMES}")

# Recreate exact participant-separated validation split
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
fit_idx, val_idx = next(gss.split(X_train_full, y_train_full, groups=subject_train_full))

X_fit, X_val = X_train_full[fit_idx], X_train_full[val_idx]
y_fit, y_val = y_train_full[fit_idx], y_train_full[val_idx]
subject_fit = subject_train_full[fit_idx]
subject_val = subject_train_full[val_idx]

fit_subjects = set(np.unique(subject_fit).tolist())
val_subjects = set(np.unique(subject_val).tolist())

# Strict zero-leakage check
assert fit_subjects.isdisjoint(val_subjects), (
    f"PARTICIPANT LEAKAGE DETECTED! Overlapping subjects: {fit_subjects.intersection(val_subjects)}"
)

print("\n" + "=" * 55)
print("PARTICIPANT-SEPARATED VALIDATION SPLIT VERIFICATION")
print("=" * 55)
print(f"Validation Windows (N_val):   {len(X_val)} samples (Shape: {X_val.shape})")
print(f"Fit Windows (N_fit):          {len(X_fit)} samples (Shape: {X_fit.shape})")
print(f"Validation Participants:      {sorted(list(val_subjects))} (Count: {len(val_subjects)})")
print(f"Fit Subset Participants:      {sorted(list(fit_subjects))} (Count: {len(fit_subjects)})")
print(f"Participant Overlap:          {fit_subjects.intersection(val_subjects)} (Zero Leakage Confirmed)")
print(f"Signal Normalization:         Signals used as-is from archive (no secondary normalization).")

# Per-activity sample counts table
validation_counts_df = pd.DataFrame({
    "Class ID": np.arange(len(ACTIVITY_NAMES)),
    "Activity": ACTIVITY_NAMES,
    "Validation Windows": np.bincount(y_val, minlength=len(ACTIVITY_NAMES)),
    "Percentage (%)": np.round(100.0 * np.bincount(y_val, minlength=len(ACTIVITY_NAMES)) / len(y_val), 2),
    "Fit Windows": np.bincount(y_fit, minlength=len(ACTIVITY_NAMES))
})
display(validation_counts_df)


## 3. Checkpoint Equivalence Verification & Safe Model Loading
Requirements:
1. Verify whether `02_Model_Training/outputs/targeted_sitting_experiment/best_targeted_cnn.keras` and `03_Final_Model/final_model/har_cnn_model_verified_candidate.keras` produce equivalent predictions.
2. Select the verified candidate checkpoint if equivalence is confirmed; otherwise use the targeted CNN checkpoint directly.
3. Verify that the selected file exists, has input shape `(None, 128, 6)`, and outputs 6 classes.
4. **Safety Guarantee:** Do not retrain, overwrite, rename, or delete any existing model checkpoints.


In [ ]:
# 3. Model Equivalence Verification and Safe Selection
print("=" * 75)
print("MODEL CHECKPOINT AUDIT & EQUIVALENCE VERIFICATION")
print("=" * 75)

targeted_exists = TARGETED_CHECKPOINT.is_file()
candidate_exists = CANDIDATE_CHECKPOINT.is_file()

print(f"Targeted CNN checkpoint path:  {TARGETED_CHECKPOINT}")
print(f"  --> File exists: {targeted_exists}")
print(f"Verified candidate path:       {CANDIDATE_CHECKPOINT}")
print(f"  --> File exists: {candidate_exists}")

if not targeted_exists and not candidate_exists:
    raise FileNotFoundError(
        "CRITICAL ERROR: Neither model checkpoint exists in the expected paths.\n"
        f"  Targeted path:  {TARGETED_CHECKPOINT}\n"
        f"  Candidate path: {CANDIDATE_CHECKPOINT}\n"
        "Execution stopped rather than silently substituting another model."
    )

equivalence_confirmed = False
chosen_checkpoint_path = None
dashboard_model = None

if targeted_exists and candidate_exists:
    print("\nLoading both checkpoints to test prediction equivalence on validation set...")
    model_targeted = tf.keras.models.load_model(TARGETED_CHECKPOINT, compile=False)
    model_candidate = tf.keras.models.load_model(CANDIDATE_CHECKPOINT, compile=False)

    # Validate shapes on both models
    for name, m in [("Targeted CNN", model_targeted), ("Candidate CNN", model_candidate)]:
        if tuple(m.input_shape[1:]) != (WINDOW_SIZE, len(CHANNEL_NAMES)):
            raise ValueError(
                f"Incompatible input shape in {name}: {m.input_shape}; "
                f"expected (None, {WINDOW_SIZE}, {len(CHANNEL_NAMES)})"
            )
        if m.output_shape[-1] != len(ACTIVITY_NAMES):
            raise ValueError(
                f"Incompatible output shape in {name}: {m.output_shape}; "
                f"expected (None, {len(ACTIVITY_NAMES)})"
            )

    # Compute validation predictions and softmax probabilities
    probs_targeted = model_targeted.predict(X_val, batch_size=128, verbose=0)
    probs_candidate = model_candidate.predict(X_val, batch_size=128, verbose=0)
    preds_targeted = np.argmax(probs_targeted, axis=1)
    preds_candidate = np.argmax(probs_candidate, axis=1)

    # Compare predictions
    same_pred_ratio = float(np.mean(preds_targeted == preds_candidate))
    max_prob_diff = float(np.max(np.abs(probs_targeted - probs_candidate)))
    identical_classes = bool(np.array_equal(preds_targeted, preds_candidate))
    close_probabilities = bool(np.allclose(probs_targeted, probs_candidate, atol=1e-5))

    print(f"\nValidation Equivalence Audit on {len(X_val)} Samples:")
    print(f"  • Same predicted class percentage:   {same_pred_ratio * 100:.4f}%")
    print(f"  • Maximum absolute probability diff: {max_prob_diff:.8f}")
    print(f"  • Exact discrete prediction match:   {identical_classes}")
    print(f"  • Softmax probabilities match < 1e-5: {close_probabilities}")

    if identical_classes and close_probabilities:
        equivalence_confirmed = True
        chosen_checkpoint_path = CANDIDATE_CHECKPOINT
        dashboard_model = model_candidate
        print("\n--> CONCLUSION: Prediction equivalence CONFIRMED (100% agreement, zero delta).")
        print(f"    Selected Checkpoint: Verified Candidate CNN ({chosen_checkpoint_path.name})")
    else:
        equivalence_confirmed = False
        chosen_checkpoint_path = TARGETED_CHECKPOINT
        dashboard_model = model_targeted
        print("\n--> CONCLUSION: Equivalence not confirmed. Using Targeted CNN checkpoint directly.")
        print(f"    Selected Checkpoint: {chosen_checkpoint_path.name}")

elif candidate_exists:
    print("\nTargeted checkpoint not found; loading verified candidate checkpoint directly...")
    chosen_checkpoint_path = CANDIDATE_CHECKPOINT
    dashboard_model = tf.keras.models.load_model(CANDIDATE_CHECKPOINT, compile=False)
else:
    print("\nCandidate checkpoint not found; loading targeted CNN checkpoint directly...")
    chosen_checkpoint_path = TARGETED_CHECKPOINT
    dashboard_model = tf.keras.models.load_model(TARGETED_CHECKPOINT, compile=False)

# Final safety checks on the loaded dashboard model
assert dashboard_model is not None, "Dashboard model failed to instantiate."
assert tuple(dashboard_model.input_shape[1:]) == (WINDOW_SIZE, len(CHANNEL_NAMES)), (
    f"Invalid input shape {dashboard_model.input_shape}; expected (None, {WINDOW_SIZE}, {len(CHANNEL_NAMES)})"
)
assert dashboard_model.output_shape[-1] == len(ACTIVITY_NAMES), (
    f"Invalid output shape {dashboard_model.output_shape}; expected (None, {len(ACTIVITY_NAMES)})"
)

print("\n" + "=" * 75)
print("ACTIVE DASHBOARD MODEL CHECKPOINT VERIFIED")
print("=" * 75)
print(f"Exact File Path:  {chosen_checkpoint_path}")
print(f"Input Shape:      {dashboard_model.input_shape} (128 time steps, 6 channels)")
print(f"Output Shape:     {dashboard_model.output_shape} (6 activity classes)")
print("Model Architecture Summary:")
dashboard_model.summary(print_fn=lambda line: print("  " + line))


## 4. Live Validation Metrics & Reporting
- Metrics are calculated **directly from predictions made by the exact checkpoint loaded above on the 1,801 validation windows**.
- Validation accuracy, macro precision, macro recall, macro F1, and weighted F1 are computed and displayed.
- **No historical metrics from other runs are copied.**
- Offline evaluation mode: benchmark dataset; not live phone sensing.


In [ ]:
# 4. Live Validation Evaluation and Metric Computation
print(f"Computing live validation predictions with {chosen_checkpoint_path.name}...")
val_probabilities = dashboard_model.predict(X_val, batch_size=128, verbose=0)
val_predictions = np.argmax(val_probabilities, axis=1)
y_true_val = np.asarray(y_val).astype(int)

# Live Metric Calculations
val_accuracy = float(accuracy_score(y_true_val, val_predictions))
val_macro_precision = float(precision_score(y_true_val, val_predictions, average="macro", zero_division=0))
val_macro_recall = float(recall_score(y_true_val, val_predictions, average="macro", zero_division=0))
val_macro_f1 = float(f1_score(y_true_val, val_predictions, average="macro", zero_division=0))
val_weighted_f1 = float(f1_score(y_true_val, val_predictions, average="weighted", zero_division=0))

# Build Summary Metrics DataFrame
live_metrics_df = pd.DataFrame([
    {"Metric": "Accuracy", "Value": val_accuracy, "Percentage": f"{val_accuracy * 100:.2f}%"},
    {"Metric": "Macro precision", "Value": val_macro_precision, "Percentage": f"{val_macro_precision * 100:.2f}%"},
    {"Metric": "Macro recall", "Value": val_macro_recall, "Percentage": f"{val_macro_recall * 100:.2f}%"},
    {"Metric": "Macro F1", "Value": val_macro_f1, "Percentage": f"{val_macro_f1 * 100:.2f}%"},
    {"Metric": "Weighted F1", "Value": val_weighted_f1, "Percentage": f"{val_weighted_f1 * 100:.2f}%"}
])

print("\n" + "=" * 60)
print(f"LIVE VALIDATION METRICS: {chosen_checkpoint_path.name}")
print("Evaluated on all 1,801 participant-separated validation windows")
print("=" * 60)
display(live_metrics_df.style.format({"Value": "{:.6f}"}))

# Detailed Per-Activity Classification Report
clf_report_dict = classification_report(
    y_true_val,
    val_predictions,
    labels=np.arange(len(ACTIVITY_NAMES)),
    target_names=ACTIVITY_NAMES,
    output_dict=True,
    zero_division=0
)
clf_report_df = pd.DataFrame(clf_report_dict).transpose()
print("\nDETAILED PER-ACTIVITY CLASSIFICATION REPORT:")
display(clf_report_df.style.format("{:.4f}"))

# Save Live Outputs to dashboard_outputs
live_metrics_df.to_csv(OUTPUT_DIR / "dashboard_validation_metrics.csv", index=False)
clf_report_df.to_csv(OUTPUT_DIR / "dashboard_validation_classification_report.csv")
print(f"\nLive evaluation outputs successfully saved to:\n  {OUTPUT_DIR}")

# Confusion Matrix Heatmap
cm = confusion_matrix(y_true_val, val_predictions, labels=np.arange(len(ACTIVITY_NAMES)))
plt.figure(figsize=(8, 6.2))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=ACTIVITY_NAMES,
    yticklabels=ACTIVITY_NAMES,
    cbar=False
)
plt.title(f"Validation Confusion Matrix — {chosen_checkpoint_path.name}", fontsize=13, pad=12)
plt.xlabel("Predicted Activity", fontsize=11, labelpad=8)
plt.ylabel("Actual Activity (Ground Truth)", fontsize=11, labelpad=8)
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. Interactive HAR Dashboard UI
### Interactive Features:
1. **Activity Filter:** Quickly select an activity category or browse all 1,801 validation windows.
2. **Validation Sample Dropdown:** Choose any individual validation sample window.
3. **Run Prediction Button:** Executes live forward pass through the loaded CNN model.
4. **Prediction Verdict Card:** Displays the predicted activity, ground truth label, confidence percentage, and an explicit match/mismatch status badge.
5. **Class Probability Bar Chart:** Displays output probabilities across all 6 activities with highlighted bars and percentage labels.
6. **Optional 6-Channel Waveform Plots:** Toggles raw signals for Body Acceleration (X/Y/Z) and Body Gyroscope (X/Y/Z) over the 2.56-second window.
7. **Time-Domain Explanation:** Describes the 128 timesteps (2.56 s @ 50 Hz) and observed signal characteristics.
8. **Student Project Notice:** Clearly disclaims offline benchmark evaluation; not live phone sensing.


In [ ]:
# 5. Interactive HAR Dashboard UI
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ---------------------------------------------------------------------------
# Helper function: explanation text for each sample
# ---------------------------------------------------------------------------
def generate_sample_explanation(actual_act: str, pred_act: str, conf: float) -> str:
    """Generates physical and time-series interpretation for the 128 timesteps."""
    is_dynamic = actual_act in ["WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS"]
    match = (actual_act == pred_act)

    explanation = (
        "<b>Sample Window Mechanics:</b><br>"
        "• <b>Duration & Sampling:</b> 128 discrete time steps captured at 50 Hz corresponds to exactly "
        "<b>2.56 seconds</b> of continuous motion (128 / 50 Hz = 2.56 s).<br>"
        "• <b>Sensors:</b> Triaxial body acceleration (X/Y/Z linear dynamics with gravity separated) "
        "and triaxial body angular velocity (X/Y/Z rotational dynamics in rad/s).<br>"
    )
    if is_dynamic:
        explanation += (
            f"• <b>Activity Dynamics ({actual_act}):</b> Dynamic gait activities display prominent periodic, "
            "rhythmic oscillations across the accelerometer channels, capturing ~2 complete footsteps. "
            "The 1D CNN filters extract distinctive frequency features and spatial correlations among axes."
        )
    else:
        explanation += (
            f"• <b>Activity Dynamics ({actual_act}):</b> Static postures (Sitting, Standing, Laying) display low-amplitude "
            "signal variance and near-zero angular velocities. Differentiation between Sitting and Standing relies on subtle "
            "postural micro-tremors, torso inclination, and baseline acceleration shifts."
        )

    if not match:
        explanation += (
            f"<br>• <b style='color:#b45309;'>Classification Note:</b> The model predicted <b>{pred_act}</b> instead of "
            f"<b>{actual_act}</b> with {conf*100:.1f}% confidence. Misclassifications most commonly occur between Sitting and Standing "
            "due to identical static gravity orientation and low dynamic variance."
        )
    return explanation

# ---------------------------------------------------------------------------
# Build Dropdown Options
# ---------------------------------------------------------------------------
def get_sample_options(filter_act="ALL"):
    """Constructs labeled dropdown items based on the selected activity filter."""
    options = []
    for i in range(len(X_val)):
        act_name = ACTIVITY_NAMES[y_val[i]]
        if filter_act == "ALL" or act_name == filter_act:
            options.append((f"Sample #{i:04d} | Actual: {act_name}", i))
    return options

# Interactive Widgets
activity_filter_dropdown = widgets.Dropdown(
    options=["ALL (All 1,801 samples)"] + ACTIVITY_NAMES,
    value="ALL (All 1,801 samples)",
    description="Activity Filter:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="340px")
)

initial_options = get_sample_options("ALL")
sample_dropdown = widgets.Dropdown(
    options=initial_options,
    value=initial_options[0][1],
    description="Validation Sample:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="440px")
)

predict_btn = widgets.Button(
    description="Run Prediction",
    button_style="primary",
    icon="play",
    tooltip="Run model inference on the selected validation sample",
    layout=widgets.Layout(width="180px", height="38px")
)

show_signals_checkbox = widgets.Checkbox(
    value=True,
    description="Show 6-Channel Sensor Waveforms",
    indent=False,
    layout=widgets.Layout(width="280px")
)

dashboard_output = widgets.Output()

# Callback: Update sample dropdown when activity filter changes
def on_filter_change(change):
    selected_filter = change["new"]
    filter_key = "ALL" if "ALL" in selected_filter else selected_filter
    new_options = get_sample_options(filter_key)
    if new_options:
        sample_dropdown.options = new_options
        sample_dropdown.value = new_options[0][1]

activity_filter_dropdown.observe(on_filter_change, names="value")

# Callback: Run prediction and render results
def run_prediction_display(sample_idx: int, show_signals: bool):
    with dashboard_output:
        clear_output(wait=True)

        # Validate sample index
        if sample_idx < 0 or sample_idx >= len(X_val):
            display(HTML(f"<div style='color:red;padding:10px;'>Error: Invalid sample index {sample_idx}.</div>"))
            return

        # Forward pass on the single selected window
        window_tensor = X_val[sample_idx:sample_idx+1]
        probs = dashboard_model.predict(window_tensor, verbose=0)[0]
        pred_class_id = int(np.argmax(probs))
        actual_class_id = int(y_val[sample_idx])

        pred_activity = ACTIVITY_NAMES[pred_class_id]
        actual_activity = ACTIVITY_NAMES[actual_class_id]
        confidence = float(probs[pred_class_id])
        is_correct = (pred_class_id == actual_class_id)

        # Verdict Badge Styling
        if is_correct:
            badge_bg = "#ecfdf5"
            badge_border = "#10b981"
            badge_text_color = "#047857"
            verdict_icon = "✅"
            verdict_text = "CORRECT PREDICTION"
        else:
            badge_bg = "#fffbeb"
            badge_border = "#f59e0b"
            badge_text_color = "#b45309"
            verdict_icon = "⚠️"
            verdict_text = "MISMATCH (DIFFERENT FROM RECORDED LABEL)"

        # Render HTML Verdict Card
        card_html = (
            f'<div style="border: 2px solid {badge_border}; background-color: {badge_bg}; border-radius: 12px; padding: 18px 24px; margin-bottom: 20px; font-family: -apple-system, BlinkMacSystemFont, sans-serif;">'
            f'  <div style="display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap;">'
            f'    <div>'
            f'      <span style="font-size: 13px; font-weight: 700; letter-spacing: 0.5px; color: #4b5563; text-transform: uppercase;">Validation Sample #{sample_idx:04d}</span>'
            f'      <h2 style="margin: 6px 0 4px 0; color: #111827; font-size: 26px;">Predicted: <b>{pred_activity}</b></h2>'
            f'      <div style="font-size: 16px; color: #374151;">Actual Recorded Activity: <b>{actual_activity}</b></div>'
            f'    </div>'
            f'    <div style="text-align: right; margin-top: 8px;">'
            f'      <div style="display: inline-block; padding: 6px 14px; border-radius: 20px; background-color: {badge_border}; color: #ffffff; font-weight: 700; font-size: 14px;">'
            f'        {verdict_icon} {verdict_text}'
            f'      </div>'
            f'      <div style="margin-top: 8px; font-size: 18px; font-weight: 700; color: #1f2937;">'
            f'        Confidence: <span style="color: {badge_text_color};">{confidence * 100:.2f}%</span>'
            f'      </div>'
            f'    </div>'
            f'  </div>'
            f'</div>'
        )
        display(HTML(card_html))

        # -------------------------------------------------------------------
        # Visualization: Probability Bar Chart & Optional Signal Plots
        # -------------------------------------------------------------------
        if show_signals:
            fig = plt.figure(figsize=(15, 8.5))
            gs = fig.add_gridspec(2, 2, width_ratios=[1.1, 1.4], hspace=0.35, wspace=0.25)
            ax_bar = fig.add_subplot(gs[:, 0])
            ax_acc = fig.add_subplot(gs[0, 1])
            ax_gyro = fig.add_subplot(gs[1, 1])
        else:
            fig, ax_bar = plt.subplots(figsize=(8, 5))

        # Probability Bar Chart
        bar_colors = []
        for i, act in enumerate(ACTIVITY_NAMES):
            if i == pred_class_id:
                bar_colors.append("#10b981" if is_correct else "#f59e0b")
            elif i == actual_class_id:
                bar_colors.append("#6366f1")  # Purple highlight for ground truth if different
            else:
                bar_colors.append("#cbd5e1")

        bars = ax_bar.barh(ACTIVITY_NAMES, probs * 100, color=bar_colors, edgecolor="#475569", height=0.6)
        ax_bar.set_xlim(0, 105)
        ax_bar.set_xlabel("Predicted Probability (%)", fontsize=11, labelpad=8)
        ax_bar.set_title("Activity Softmax Probabilities", fontsize=13, weight="bold", pad=12)
        ax_bar.invert_yaxis()

        # Add percentage text labels on bars
        for bar, prob in zip(bars, probs):
            width = bar.get_width()
            ax_bar.text(
                width + 1.5,
                bar.get_y() + bar.get_height() / 2,
                f"{prob * 100:.1f}%",
                va="center",
                ha="left",
                fontsize=10,
                weight="bold" if prob == probs.max() else "normal",
                color="#111827"
            )

        # Signal Waveforms (if toggled)
        if show_signals:
            time_axis = np.arange(WINDOW_SIZE) / SAMPLING_RATE_HZ  # 0 to 2.56 seconds
            sample_signal = X_val[sample_idx]  # shape (128, 6)

            # Acceleration Subplot (Channels 0, 1, 2)
            ax_acc.plot(time_axis, sample_signal[:, 0], label="Body Acc X (lateral)", color="#2563eb", lw=1.6)
            ax_acc.plot(time_axis, sample_signal[:, 1], label="Body Acc Y (vertical)", color="#16a34a", lw=1.6)
            ax_acc.plot(time_axis, sample_signal[:, 2], label="Body Acc Z (forward)", color="#dc2626", lw=1.6)
            ax_acc.set_title("Body Acceleration Channels (X / Y / Z)", fontsize=11, weight="bold")
            ax_acc.set_ylabel("Normalized Acc", fontsize=10)
            ax_acc.set_xlim(0, WINDOW_DURATION_SEC)
            ax_acc.grid(True, linestyle="--", alpha=0.6)
            ax_acc.legend(loc="upper right", fontsize=8.5, framealpha=0.9)

            # Gyroscope Subplot (Channels 3, 4, 5)
            ax_gyro.plot(time_axis, sample_signal[:, 3], label="Body Gyro X (pitch)", color="#7c3aed", lw=1.6)
            ax_gyro.plot(time_axis, sample_signal[:, 4], label="Body Gyro Y (roll)", color="#d97706", lw=1.6)
            ax_gyro.plot(time_axis, sample_signal[:, 5], label="Body Gyro Z (yaw)", color="#0891b2", lw=1.6)
            ax_gyro.set_title("Body Gyroscope Channels (X / Y / Z)", fontsize=11, weight="bold")
            ax_gyro.set_xlabel("Time (seconds) — [128 timesteps @ 50 Hz = 2.56 s]", fontsize=10, labelpad=6)
            ax_gyro.set_ylabel("Normalized Gyro", fontsize=10)
            ax_gyro.set_xlim(0, WINDOW_DURATION_SEC)
            ax_gyro.grid(True, linestyle="--", alpha=0.6)
            ax_gyro.legend(loc="upper right", fontsize=8.5, framealpha=0.9)

        plt.tight_layout()
        plt.show()

        # -------------------------------------------------------------------
        # Physical Interpretation & Explanation Box
        # -------------------------------------------------------------------
        explanation_text = generate_sample_explanation(actual_activity, pred_activity, confidence)
        explanation_html = (
            f'<div style="background-color: #f8fafc; border-left: 4px solid #3b82f6; border-radius: 4px 8px 8px 4px; padding: 14px 18px; margin-top: 15px; font-size: 13.5px; line-height: 1.6; color: #334155;">'
            f'  {explanation_text}'
            f'</div>'
            f'<div style="margin-top: 10px; font-size: 12px; color: #64748b; font-style: italic;">'
            f'  <b>Demonstration Notice:</b> Offline evaluation on recorded benchmark dataset (UCI HAR); this dashboard does NOT perform live phone sensing or real-time physical streaming.'
            f'</div>'
        )
        display(HTML(explanation_html))

# Button Click Event Listener
def on_predict_button_clicked(b):
    run_prediction_display(sample_dropdown.value, show_signals_checkbox.value)

predict_btn.on_click(on_predict_button_clicked)

# Render Controls Layout
controls_ui = widgets.VBox([
    widgets.HTML(
        '<div style="background: linear-gradient(135deg, #1e3a8a, #3b82f6); color: white; padding: 14px 20px; border-radius: 10px; margin-bottom: 12px;">'
        '  <h3 style="margin: 0; color: white;">Interactive Human Activity Recognition Dashboard</h3>'
        '  <p style="margin: 4px 0 0 0; font-size: 13px; opacity: 0.9;">'
        '    Select any validation window or filter by activity, then click <b>Run Prediction</b> to evaluate the loaded CNN model.'
        '  </p>'
        '</div>'
    ),
    widgets.HBox([activity_filter_dropdown, sample_dropdown], layout=widgets.Layout(margin="0 0 10px 0")),
    widgets.HBox([predict_btn, show_signals_checkbox], layout=widgets.Layout(align_items="center", margin="0 0 15px 0")),
    dashboard_output
])

display(controls_ui)

# Trigger initial display on default sample
run_prediction_display(sample_dropdown.value, show_signals_checkbox.value)


## 6. Programmatic Sample Prediction API
For convenience, this function can also be invoked programmatically from standard Python code cells with any valid validation index (0 to 1,800).


In [ ]:
# 6. Programmatic Sample Prediction Function
def predict_sample(sample_index: int, show_signals: bool = True):
    """
    Performs inference on a specific validation sample by index and returns a summary dictionary.
    Parameters:
        sample_index (int): Index between 0 and len(X_val) - 1.
        show_signals (bool): Whether to plot the 6-channel sensor waveforms.
    Returns:
        dict: Summary of prediction, probabilities, actual label, and correctness.
    """
    if not (0 <= sample_index < len(X_val)):
        raise IndexError(f"Sample index {sample_index} out of bounds. Valid range: 0 to {len(X_val) - 1}.")

    sample_tensor = X_val[sample_index:sample_index+1]
    probabilities = dashboard_model.predict(sample_tensor, verbose=0)[0]
    predicted_id = int(np.argmax(probabilities))
    actual_id = int(y_val[sample_index])

    result = {
        "sample_index": sample_index,
        "predicted_activity": ACTIVITY_NAMES[predicted_id],
        "actual_activity": ACTIVITY_NAMES[actual_id],
        "confidence": float(probabilities[predicted_id]),
        "is_correct": bool(predicted_id == actual_id),
        "probabilities": {act: float(p) for act, p in zip(ACTIVITY_NAMES, probabilities)}
    }

    # Run visual display
    run_prediction_display(sample_index, show_signals)
    return result

# Example programmatic call on validation sample 42
_ = predict_sample(42, show_signals=False)


## 7. Model Checkpoint & Validation Metrics Summary
This section documents the exact checkpoint used, the live validation metrics obtained, and instructions for full reproducibility.


In [ ]:
# 7. Model Checkpoint & Validation Metrics Summary
print("=" * 78)
print("HUMAN ACTIVITY RECOGNITION (HAR) — FINAL DASHBOARD SUMMARY")
print("=" * 78)
print(f"Active Checkpoint:          {chosen_checkpoint_path}")
print(f"Model Type:                 Targeted Class-Weighted 1D CNN")
print(f"Input Shape:                {dashboard_model.input_shape}")
print(f"Output Shape:               {dashboard_model.output_shape} (6 classes)")
print(f"Validation Partition:       Participant-Separated (test_size=0.20, seed={SEED})")
print(f"Validation Samples:         {len(X_val)}")
print("-" * 78)
print(f"Validation Accuracy:        {val_accuracy * 100:.2f}% ({val_accuracy:.6f})")
print(f"Macro Precision:            {val_macro_precision * 100:.2f}% ({val_macro_precision:.6f})")
print(f"Macro Recall:               {val_macro_recall * 100:.2f}% ({val_macro_recall:.6f})")
print(f"Macro F1 Score:             {val_macro_f1 * 100:.2f}% ({val_macro_f1:.6f})")
print(f"Weighted F1 Score:          {val_weighted_f1 * 100:.2f}% ({val_weighted_f1:.6f})")
print("=" * 78)
print("REPRODUCIBILITY CONFIRMATION:")
print("1. All validation metrics above were computed live from the loaded checkpoint.")
print("2. No model training, renaming, or modification occurred.")
print("3. Restarting runtime and executing from top to bottom reproduces identical outputs.")
print("=" * 78)
